In [ ]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Фиксируем seed для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

In [ ]:
dataset = load_dataset("emotion")

print("Размеры датасета:")
print(dataset)

# Посмотрим на классы
print("\nКлассы:", dataset["train"].features["label"].names)

# Примеры
print("\nПримеры текстов:")
for i in range(5):
    print(f"{i+1}. {dataset['train'][i]['text'][:150]}... → {dataset['train'].features['label'].int2str(dataset['train'][i]['label'])}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Пример токенизации
texts = [
    "I am so happy today!",
    "This movie was terrible and boring.",
    "I feel sad and lonely."
]

print("Примеры токенизации:\n")
for text in texts:
    tokens = tokenizer(text, padding=True, truncation=True, max_length=64, return_tensors="pt")
    print(f"Текст: {text}")
    print(f"Токены: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
    print(f"input_ids: {tokens['input_ids'][0].tolist()}")
    print(f"attention_mask: {tokens['attention_mask'][0].tolist()}")
    print("-" * 70)

In [ ]:
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)

# Простой инференс
print("Инференс предобученной модели (случайные веса классификатора):\n")

for i in range(5):
    text = dataset["test"][i]["text"]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    true_label = dataset["test"].features["label"].int2str(dataset["test"][i]["label"])
    pred_label = dataset["test"].features["label"].int2str(pred)

    print(f"Текст: {text[:120]}...")
    print(f"Истинно: {true_label} | Предсказано: {pred_label}\n")

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",           # современное название
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=42,
    report_to="none",
    logging_steps=50,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    # tokenizer больше не передаётся в Trainer (убрали)
)

print("Начинаем обучение...")
trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print("Результаты на test:")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"F1-macro:  {test_results['eval_f1']:.4f}")

In [ ]:
import os
os.makedirs('artifacts/figures', exist_ok=True)


In [ ]:
predictions_output = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

# Преобразуем метки в названия классов (исправленный вариант)
label_feature = dataset["train"].features["label"]
class_names = label_feature.names

# Надёжное преобразование в строки
true_label_names = [class_names[int(label)] for label in true_labels[:100]]
pred_label_names = [class_names[int(label)] for label in pred_labels[:100]]

# Создаём DataFrame
df = pd.DataFrame({
    "text": dataset["test"]["text"][:100],
    "true_label": true_label_names,
    "pred_label": pred_label_names
})

# Сохраняем
df.to_csv("artifacts/sample_predictions.csv", index=False)
print(f"Сохранено {len(df)} примеров в sample_predictions.csv")

cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix on Test Set')
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png")
plt.show()

print("Артефакты успешно сохранены!")
print(f"Accuracy на test: {accuracy_score(true_labels, pred_labels):.4f}")
print(f"F1-macro на test: {f1_score(true_labels, pred_labels, average='macro'):.4f}")